<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/JAX_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, warnings, logging

# --- 0. SUPPRESS LOGS ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['JAX_LOG_LEVEL'] = '0'

# Specifically target the Transparent Hugepages warning
warnings.filterwarnings("ignore", message="Transparent hugepages are not enabled")

# General suppressions
warnings.filterwarnings('ignore')
logging.getLogger('jax').setLevel(logging.ERROR)

In [2]:
import os, warnings, jax, optax, requests
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import numpy as np

# --- 1. SETTINGS & DATA ---
SEED, BLOCK_SIZE, EMBED_DIM = 123, 128, 384
NUM_HEADS, NUM_LAYERS, BATCH_SIZE_PER_CORE = 6, 6, 32
TOTAL_STEPS = 5001
devices = jax.local_devices()
BATCH_SIZE = BATCH_SIZE_PER_CORE * len(devices)

def get_data():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    text = requests.get(url).text
    chars = sorted(list(set(text)))
    mapping = {ch: i for i, ch in enumerate(chars)}
    return text, chars, mapping

text, chars, char_to_int = get_data()
VOCAB_SIZE = len(chars)
encode = lambda s: [char_to_int[c] for c in s]
decode = lambda l: ''.join([chars[i] for i in l])

# --- 2. MODERN COMPONENTS (RoPE & RMSNorm) ---
class RMSNorm(nn.Module):
    @nn.compact
    def __call__(self, x):
        scale = self.param('scale', jax.nn.initializers.ones, (x.shape[-1],))
        # RMSNorm is faster and more stable than LayerNorm
        var = jnp.mean(jnp.square(x), axis=-1, keepdims=True)
        return x * jax.lax.rsqrt(var + 1e-6) * scale

def apply_rope(xq, xk):
    # Rotary Positional Embeddings (RoPE)
    dim = xq.shape[-1]
    freqs = 1.0 / (10000**(jnp.arange(0, dim, 2)[:(dim // 2)] / dim))
    t = jnp.arange(xq.shape[2])
    freqs = jnp.outer(t, freqs)
    freqs_cis = jnp.cos(freqs), jnp.sin(freqs)

    def rotate(x, cos, sin):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return jnp.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], axis=-1).reshape(x.shape)

    return rotate(xq, *freqs_cis), rotate(xk, *freqs_cis)

# --- 3. MODEL ---
class ModernGPT(nn.Module):
    @nn.compact
    def __call__(self, idx, mask):
        B, T = idx.shape
        # We tie weights between embedding and output head
        tok_emb = nn.Embed(VOCAB_SIZE, EMBED_DIM, name="embed")
        x = tok_emb(idx)

        for _ in range(NUM_LAYERS):
            # Attention with RoPE
            res = x
            x = RMSNorm()(x)
            qkv = nn.Dense(3 * EMBED_DIM, use_bias=False)(x)
            q, k, v = [t.reshape(B, T, NUM_HEADS, -1).transpose(0, 2, 1, 3) for t in jnp.split(qkv, 3, axis=-1)]
            q, k = apply_rope(q, k)

            #
            attn = jax.nn.softmax((q @ k.transpose(0, 1, 3, 2)) / jnp.sqrt(q.shape[-1]) + mask)
            out = (attn @ v).transpose(0, 2, 1, 3).reshape(B, T, EMBED_DIM)
            x = res + nn.Dense(EMBED_DIM, use_bias=False)(out)

            # Feed-Forward
            res = x
            x = RMSNorm()(x)
            h = nn.Dense(4 * EMBED_DIM, use_bias=False)(x)
            x = res + nn.Dense(EMBED_DIM, use_bias=False)(jax.nn.gelu(h))

        x = RMSNorm()(x)
        return x @ tok_emb.variables['params']['embedding'].T

# --- 4. TRAINING LOGIC ---
#lr_sched = optax.warmup_cosine_decay_schedule(0, 5e-4, 500, TOTAL_STEPS)

lr_sched = 5e-4
print(f"Learning Rate: {lr_sched}")

#
state = train_state.TrainState.create(
    apply_fn=ModernGPT().apply,
    params=ModernGPT().init(jax.random.PRNGKey(0), jnp.ones((1, BLOCK_SIZE), jnp.int32), jnp.ones((BLOCK_SIZE, BLOCK_SIZE)))['params'],
    tx=optax.adamw(lr_sched)
)

def train_step(state, bx, by, mask):
    grad_fn = jax.value_and_grad(lambda p: optax.softmax_cross_entropy_with_integer_labels(state.apply_fn({'params': p}, bx, mask), by).mean())
    loss, grads = grad_fn(state.params)
    return state.apply_gradients(grads=jax.lax.pmean(grads, "b")), jax.lax.pmean(loss, "b")

p_train = jax.pmap(train_step, "b")
state = jax.device_put_replicated(state, devices)
mask = jax.device_put_replicated(jnp.triu(jnp.full((BLOCK_SIZE, BLOCK_SIZE), -1e10), 1), devices)
data_idx = jnp.array(encode(text))

# --- 5. LOOP & GENERATE ---
devices = jax.local_devices()
num_devices = len(devices)

print("-"*40)
print(f"TPU Training Started | Cores: {num_devices}")
print(f"Batch Size: {BATCH_SIZE} | Block Size: {BLOCK_SIZE}")
print(f"Total Steps: {TOTAL_STEPS}")
print("-"*40)
print("\n")


rng = jax.random.PRNGKey(SEED)
for i in range(TOTAL_STEPS):
    rng, _rng = jax.random.split(rng)
    starts = jax.random.randint(_rng, (BATCH_SIZE,), 0, len(data_idx)-BLOCK_SIZE-1)
    bx = jnp.stack([data_idx[s:s+BLOCK_SIZE] for s in starts]).reshape(len(devices), BATCH_SIZE_PER_CORE, -1)
    by = jnp.stack([data_idx[s+1:s+BLOCK_SIZE+1] for s in starts]).reshape(len(devices), BATCH_SIZE_PER_CORE, -1)
    state, loss = p_train(state, bx, by, mask)
    if i % 500 == 0: print(f"Step {i} | Loss: {loss[0]:.4f}")

def generate_top_k(state, start_str, length=4096, k=10):
    params = jax.tree_util.tree_map(lambda x: x[0], state.params)
    tokens = jnp.array([encode(start_str)])
    gen_rng = jax.random.PRNGKey(SEED)
    for _ in range(length):
        inp = tokens[:, -BLOCK_SIZE:]
        m = jnp.triu(jnp.full((inp.shape[1], inp.shape[1]), -1e10), 1)
        logits = ModernGPT().apply({'params': params}, inp, m)[0, -1, :]
        val, idx = jax.lax.top_k(logits, k)
        gen_rng, _ = jax.random.split(gen_rng)
        next_tok = idx[jax.random.categorical(_, val)]
        tokens = jnp.concatenate([tokens, jnp.array([[next_tok]])], axis=1)
    return decode(tokens[0].tolist())


print("\n" + "="*40)
print(f"GENERATED RESPONSE (Loss: {float(loss[0]):.4f})")
print(generate_top_k(state, "KING RICHARD: "))
print("="*40)


Learning Rate: 0.0005
----------------------------------------
TPU Training Started | Cores: 1
Batch Size: 32 | Block Size: 128
Total Steps: 5001
----------------------------------------


Step 0 | Loss: 4.7828
Step 500 | Loss: 1.5317
Step 1000 | Loss: 1.3997
Step 1500 | Loss: 1.2954
Step 2000 | Loss: 1.2494
Step 2500 | Loss: 1.1542
Step 3000 | Loss: 1.0708
Step 3500 | Loss: 1.0753
Step 4000 | Loss: 0.9522
Step 4500 | Loss: 0.8563
Step 5000 | Loss: 0.8277

GENERATED RESPONSE (Loss: 0.8277)
KING RICHARD: I muse me say; saw then I defy:
The better can make them for my purse against them,
For mischiefs made to his ulteous day.

LUCENTIO:
I call them for here in her comfort.

VIRGILIA:
O my lord perhaps, I shall be help their pedness them;
for their deservail with one already, bold, bow,
Swelling, or one word, so discrettly, one summer,
What is your doing?

PETRUCHIO:
I mean, sir, be it that you miss afeth,
As I by such as mighty driven in boscmed
The fledition of his life's love and heart